# Phage σ70 Promoter → Genome Slicer (white-box)

目標：把 host-RNAP-dependent 的 phage promoter，從**你親自確認過的 TSS**，切出全長 window 與六元件（UP / -35 / spacer / -10 / disc / ITS），輸出成 CSV。

**分工**
- 機械部分（這個 notebook 做）：下載 genome、依 +1 座標切窗、reverse-complement、標元件、輸出 CSV、做 consensus self-check。
- 判斷部分（你做）：每條 promoter 的 +1 在哪、在哪條 strand、出處是哪篇論文哪張圖/表。**notebook 不替你猜任何座標。**

**三種把 +1 釘到 genome 的方法（每條 promoter 擇一）**
1. `manual`：你從論文/GenBank feature 讀到 +1 的 genome 座標 → 填 `tss_pos`（1-based，和 NCBI 顯示一致）。
2. `anchor`：你從論文貼一段**公布過的精確序列**（例如 -35..-10 那段或 footprint probe），並標明它第一個 base 對應的 promoter 座標 → notebook 去 genome 兩股搜、反推 +1。不需要絕對座標。
3. 先用 `inspect_features()` 把 GenBank 裡標註的 promoter feature 印出來挑，再回到 1 或 2。

座標慣例：promoter 座標沒有 0（… -2, -1, +1, +2 …），+1 = TSS 第一個被轉錄的 base。


## Step 1 — 文獻溯源地圖（起點，務必自行核對確切 TSS）

下面是 E. coli 端的候選與查 TSS 的起點。圖表編號我不臆造——請打開來源確認 +1 的確切位置/座標。

| Promoter | phage | genome (RefSeq/GenBank，需自行核對版本) | 查 TSS / 序列的起點 |
|---|---|---|---|
| A1, A2, A3 | T7 | NC_001604 / V01146 | 早期三 promoter 序列最早由 1978 年工作定出；完整基因組註解見 Dunn & Studier 1983 (J Mol Biol)。RefSeq 常把早期 promoter 標成 regulatory feature，可用 `inspect_features()` 撈，再對原文。|
| N25 | T5 | T5 完整基因組（自行於 NCBI 搜 'Escherichia phage T5 complete genome' 取 accession） | Brunner & Bujard 1987；ITS/escape 特性見 Hsu & Han 2025 JBC (doi:10.1016/j.jbc.2025.110610)。N25 是 -10/-35 class 模型 promoter，建議用 `anchor` 法貼公布序列最穩。|
| PR, PL, PRM | λ | NC_001416 / J02459 | 經典 σ70，序列列於 Hawley & McClure 1983 (Nucleic Acids Res) 的 promoter compilation；λ 基因組註解也含 regulatory features。注意 PR/PL 受 CI/Cro 調控。|
| early (class I) | T4 | T4 完整基因組（NCBI 搜 'Enterobacteria phage T4 complete genome'） | 只取 early（host σ70）；中期 promoter 需 MotA/AsiA，排除。|

> 自行核對的重點：(a) +1 的 genome 座標或可 anchor 的序列；(b) strand；(c) 該 promoter 是純宿主 RNAP、不需 phage activator。


In [1]:
# === Path bootstrap (shared by 01-07) ===
# Locates MS2_Data_PyTorch/scripts/library_release by walking up from the cwd,
# then imports _paths, which sets every other path absolutely and puts
# MS2_Data_PyTorch/scripts on sys.path. Safe to run from any working directory.
import sys
from pathlib import Path

for _c in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    _rel = _c / "MS2_Data_PyTorch" / "scripts" / "library_release"
    if (_rel / "_paths.py").exists():
        if str(_rel) not in sys.path:
            sys.path.insert(0, str(_rel))
        break
else:
    raise RuntimeError(f"library_release not found from {Path.cwd()}")

from _paths import *  # noqa: F401,F403

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_DIR     :", DATA_DIR)
print("RELEASE_OUT  :", RELEASE_OUT)


PROJECT_ROOT : C:\project\Whole-model
DATA_DIR     : C:\project\Whole-model\MS2_Data_PyTorch\scripts\library_release\data
RELEASE_OUT  : C:\project\Whole-model\MS2_Data_PyTorch\scripts\library_release\outputs


In [2]:
# === 環境 ===
# 需在有網路的機器上跑（你的 laptop / iris）。第一次請安裝：
#   pip install biopython pandas
from Bio import Entrez, SeqIO
from Bio.Seq import Seq
import pandas as pd
from pathlib import Path
import sys

Entrez.email = 'FILL_YOUR_EMAIL@example.com'   # NCBI 要求；不填會被擋
# Entrez.api_key = 'xxxx'                       # 有 API key 可解除速率限制（選填）

# Paths come from _paths (bootstrap cell above): the same data/phage folder
# 03 scans, and the shared outputs/ folder. BPM resolves through
# MS2_Data_PyTorch/scripts, already on sys.path.
DATA = PHAGE_DIR    # genomes are cached here; downloads land here too
OUT = RELEASE_OUT   # CSV output

from BPM import BPM as bpm


In [3]:
# === 候選清單：你要編輯的主表 ===
# 每條擇一 method：'manual'(填 tss_pos) 或 'anchor'(填 anchor_seq + anchor_start_coord)。
# 還是 TODO 的條目，主迴圈會跳過並印出缺什麼，不會 crash。
#
# 欄位說明：
#   accession         : genome 的 NCBI accession（請自行核對版本號）
#   strand            : '+' 或 '-'（promoter 轉錄方向；manual 法必填）
#   method            : 'manual' | 'anchor'
#   tss_pos           : (manual) +1 的 genome 座標，1-based，和 NCBI 顯示一致
#   anchor_seq        : (anchor) 論文公布的精確片段（5'->3'，promoter 方向）
#   anchor_start_coord: (anchor) anchor_seq 第一個 base 的 promoter 座標（如 -38）
#   location_ok       : 選填，預設 True。False = 這條的 +1 推導不可信，會被
#                       標成 qc_pass=False，06 預設不收。
#
# accession 同時是 data/phage/ 下的檔名 key，所以維持磁碟上的寫法；輸出表記的是
# GenBank record.id（帶版本號），這樣才和 03 對得起來。
#
# T5 promoter.xlsx 的第二欄是 GenBank regulatory feature location。
# 這些 location 的外側 upstream 邊界可視為 full window 的 -60 端；
# 即使 feature 長度超過或短於 (-60..+15)，仍用最外側邊界反推 +1。

TODO = None  # 哨兵：尚未填
T5_FULL_WINDOW = (-60, 15)  # 必須和下一個 cell 的 ELEMENTS['full'] 保持一致

T5_PROMOTER_LOCATIONS = [
    ('P-D/E 20', 'complement(4338..4664)'),
    ('P-H 22', 'complement(4799..4873)'),
    ('P-F 30', 'complement(16779..16784)'),
    ('P-D/E 33', 'complement(42227..42301)'),
    ('P-H 207', 'complement(45985..46059)'),
    ('P-N 25', '63043..63117'),
    ('P-N 26', '64068..64142'),
    ('P-K 28b', '66366..66440'),
    ('P-K 28a', '66459..66533'),
    ('P-G 25', 'complement(92013..92087)'),
    ('P-J 5', 'complement(99910..99984)'),
]

def parse_genbank_location(location):
    """Parse simple GenBank locations like 63043..63117 or complement(92013..92087)."""
    loc = location.strip()
    strand = '-'
    if loc.startswith('complement(') and loc.endswith(')'):
        loc = loc[len('complement('):-1]
    else:
        strand = '+'
    start, end = [int(x) for x in loc.split('..')]
    return start, end, strand

def tss_from_full_window_location(location, full_window):
    """Infer +1 from the trusted upstream edge of a T5 promoter feature.

    Returns (tss_pos, strand, note, location_ok). The inference only holds if the
    feature really spans the nominal window, so a length mismatch is reported as
    location_ok=False rather than as a note nobody reads: two entries in
    'T5 promoter.xlsx' are not 75 bp (P-D/E 20 is 327, P-F 30 is 6), and their
    +1 cannot be trusted.
    """
    start, end, strand = parse_genbank_location(location)
    a, b = full_window
    if strand == '+':
        tss_pos = start - a
    else:
        tss_pos = end + a
    feature_len = end - start + 1
    expected_len = abs(a) + b
    location_ok = feature_len == expected_len
    note = f'feature length {feature_len}; inferred +1 from upstream edge at promoter coord {a}'
    if not location_ok:
        note += f' (not equal to nominal full window length {expected_len}; +1 NOT trusted)'
    return tss_pos, strand, note, location_ok

def t5_promoter_candidates():
    rows = []
    for label, location in T5_PROMOTER_LOCATIONS:
        tss_pos, strand, warning, location_ok = tss_from_full_window_location(
            location, T5_FULL_WINDOW)
        safe_label = label.replace('P-', '').replace('/', '_').replace(' ', '_')
        rows.append(dict(name=f'T5_{safe_label}', phage='T5', host='E.coli', sigma='sigma70',
                         accession='AY543070.1', strand=strand, method='manual',
                         location_ok=location_ok,
                         anchor_seq=TODO, anchor_start_coord=TODO, tss_pos=tss_pos,
                         source='data/T5 promoter.xlsx; AY543070.1 GenBank regulatory feature',
                         notes=f'promoter {label}; location={location}; {warning}'))
    return rows

CANDIDATES = [
    dict(name='T7_A1', phage='T7', host='E.coli', sigma='sigma70',
         accession='NC_001604.1', strand='+', method='manual',
         anchor_seq=TODO, anchor_start_coord=TODO, tss_pos=498,
         source='Dunn&Studier 1983; 1978 early-promoter seq', notes='major early promoter'),
    dict(name='T7_A2', phage='T7', host='E.coli', sigma='sigma70',
         accession='NC_001604.1', strand='+', method='manual',
         anchor_seq=TODO, anchor_start_coord=TODO, tss_pos=626,
         source='1978 early-promoter seq', notes='A2/A3 share 17bp in -35 region'),
    dict(name='T7_A3', phage='T7', host='E.coli', sigma='sigma70',
         accession='NC_001604.1', strand='+', method='manual',
         anchor_seq=TODO, anchor_start_coord=TODO, tss_pos=750,
         source='1978 early-promoter seq', notes=''),
    dict(name='T7_B', phage='T7', host='E.coli', sigma='sigma70',
         accession='NC_001604.1', strand='+', method='manual',
         anchor_seq=TODO, anchor_start_coord=TODO, tss_pos=1514,
         source='1978 early-promoter seq', notes=''),
    dict(name='T7_C', phage='T7', host='E.coli', sigma='sigma70',
         accession='NC_001604.1', strand='+', method='manual',
         anchor_seq=TODO, anchor_start_coord=TODO, tss_pos=3113,
         source='1978 early-promoter seq', notes=''),
    dict(name='T7_E[6]', phage='T7', host='E.coli', sigma='sigma70',
         accession='NC_001604.1', strand='+', method='manual',
         anchor_seq=TODO, anchor_start_coord=TODO, tss_pos=36836,
         source='1978 early-promoter seq', notes=''),
    *t5_promoter_candidates(),
    dict(name='lambda_PR', phage='lambda', host='E.coli', sigma='sigma70',
         accession='NC_001416', strand='+', method='manual',
         tss_pos=38023, anchor_seq=TODO, anchor_start_coord=TODO,
         source='Hawley&McClure 1983', notes='CI/Cro-regulated; 需無 CI 背景才 constitutive'),
    dict(name='lambda_PL', phage='lambda', host='E.coli', sigma='sigma70',
         accession='NC_001416', strand='-', method='manual',
         tss_pos=34560, anchor_seq=TODO, anchor_start_coord=TODO,
         source='Hawley&McClure 1983', notes='N'),
    dict(name='L5_Pleft', phage='L5', host='M.smegmatis', sigma='SigA',
         accession='NC_001335', strand='-', method='manual',
         tss_pos=51672, anchor_seq=None, anchor_start_coord=None,
         source='Brown 1997 EMBO; Dedrick 2017 BMC Microbiol',
         notes='NAN'),
    dict(name='D29_Pleft', phage='D29', host='M.smegmatis', sigma='SigA',
         accession='NC_001900.2', strand='-', method='manual',
         tss_pos=48503, anchor_seq=None, anchor_start_coord=None,
         source='Brown 1997 EMBO; Dedrick 2017 BMC Microbiol',
         notes='NAN'),
]






In [4]:
# === 元件 window（相對 +1 的 promoter 座標，a..b 皆 inclusive）===
# 注意：-35 box 的位置假設 spacer≈17bp；真實 promoter spacer 會變動，
# 這裡是 first-pass 切法，-35/-10 邊界請用你的 scanner 或人工再 refine。
ELEMENTS = {
    # 'UP':      (-60, -40),
    # 'minus35': (-36, -31),
    # 'spacer':  (-30, -13),
    # 'minus10': (-12,  -7),
    # 'disc':    ( -6,  -1),
    # 'ITS':     (  1,  20),
    'full':    T5_FULL_WINDOW,
}
CONSENSUS = {'minus35': 'TTGACA', 'minus10': 'TATAAT'}



In [5]:
# === 核心函式（已用 toy genome 對 plus/minus 兩股驗證過）===
def coord_to_gi(tss0, strand, c):
    """promoter 座標 c -> 0-based genome index；tss0 為 +1 base 的 0-based index"""
    if c == 0:
        raise ValueError('promoter coord 0 不存在')
    offset = (c - 1) if c > 0 else c
    return tss0 + offset if strand == '+' else tss0 - offset

def slice_region(genome, tss0, strand, a, b):
    """回傳 promoter 方向（5'->3'）的序列，座標 a<b inclusive"""
    gi_a = coord_to_gi(tss0, strand, a)
    gi_b = coord_to_gi(tss0, strand, b)
    lo, hi = sorted((gi_a, gi_b))
    if lo < 0 or hi >= len(genome):
        raise IndexError('切窗超出 genome 邊界，檢查 tss_pos/strand')
    sub = genome[lo:hi + 1]
    return sub if strand == '+' else str(Seq(sub).reverse_complement())

def fetch_genome(accession):
    """下載 GenBank 檔（含序列與 feature），快取於 data/。回傳 SeqRecord"""
    gb = DATA / f'{accession}.gb'
    if not gb.exists():
        print(f'  下載 {accession} ...')
        with Entrez.efetch(db='nuccore', id=accession, rettype='gbwithparts', retmode='text') as h:
            gb.write_text(h.read())
    return SeqIO.read(gb, 'genbank')

def inspect_features(record, kinds=('regulatory','promoter','misc_feature')):
    """印出可能是 promoter 的 features，供你挑座標填回 manual"""
    for f in record.features:
        if f.type in kinds:
            note = f.qualifiers.get('note', f.qualifiers.get('regulatory_class', ['']))
            print(f.type, int(f.location.start)+1, int(f.location.end),
                  f.location.strand, '|', ';'.join(note))

def tss0_from_anchor(genome_str, anchor_seq, anchor_start_coord):
    """用論文公布序列當錨點，反推 +1 的 0-based index 與 strand"""
    a = anchor_seq.upper(); off0 = (anchor_start_coord-1) if anchor_start_coord>0 else anchor_start_coord
    p = genome_str.find(a)
    if p != -1:
        return p - off0, '+'
    rc = str(Seq(a).reverse_complement()); q = genome_str.find(rc)
    if q != -1:
        first_base_gi = q + len(a) - 1            # anchor 5' 端落在 plus 股的高位 index
        return first_base_gi + off0, '-'
    raise ValueError('anchor 在 genome 兩股都找不到，檢查序列是否抄錯')

BPM_SPACER_LENGTHS = [15, 16, 17, 18, 19]

def scan_bpm_in_sequence(seq, spacer_lengths=BPM_SPACER_LENGTHS):
    """在 promoter 方向序列內掃 BPM core promoter，回傳最高 pred_logexp hit。"""
    seq = str(seq).upper()
    hits = []
    for spacer_len in spacer_lengths:
        core_len = 6 + spacer_len + 6
        for start0 in range(0, len(seq) - core_len + 1):
            core = seq[start0:start0 + core_len]
            if set(core) - {'A', 'C', 'G', 'T'}:
                continue
            minus35, spacer, minus10 = bpm.promoter_elements(core)
            dG35, dGsp, dG10 = bpm.score_promoter_elements(core)
            dG_total = dG35 + dGsp + dG10
            hits.append({
                'bpm_core_start_in_full_1based': start0 + 1,
                'bpm_core_end_in_full_1based': start0 + core_len,
                'bpm_spacer_len': spacer_len,
                'bpm_minus35': minus35,
                'bpm_spacer': spacer,
                'bpm_minus10': minus10,
                'bpm_core_seq': core,
                'bpm_dG_minus35': dG35,
                'bpm_dG_spacer': dGsp,
                'bpm_dG_minus10': dG10,
                'bpm_dG_total': dG_total,
                'bpm_pred_exp': bpm.score2exp(dG_total),
                'bpm_pred_logexp': bpm.score2logexp(dG_total),
            })
    if not hits:
        return {}
    return max(hits, key=lambda x: x['bpm_pred_logexp'])


In [6]:
# === 主流程：逐條建 row ===
def is_todo(*vals): return any(v is None for v in vals)

rows, genome_cache = [], {}
for c in CANDIDATES:
    
    name = c['name']
    # 取 genome
    if c['accession'] is None:
        print(f'[skip] {name}: 缺 accession'); continue
    if c['accession'] not in genome_cache:
        _rec = fetch_genome(c['accession'])
        genome_cache[c['accession']] = (str(_rec.seq).upper(), _rec.id)
    gseq, record_id = genome_cache[c['accession']]
    # 定 tss0 + strand
    try:
        if c['method'] == 'manual':
            if is_todo(c['tss_pos'], c['strand']):
                print(f'[skip] {name}: manual 需 tss_pos + strand'); continue
            tss0, strand = c['tss_pos'] - 1, c['strand']
        elif c['method'] == 'anchor':
            if is_todo(c['anchor_seq'], c['anchor_start_coord']):
                print(f'[skip] {name}: anchor 需 anchor_seq + anchor_start_coord'); continue
            tss0, strand = tss0_from_anchor(gseq, c['anchor_seq'], c['anchor_start_coord'])
            # self-check：用反推的 tss0 重切 anchor 必須一致
            a0 = c['anchor_start_coord']; a1 = a0 + len(c['anchor_seq']) - 1
            assert slice_region(gseq, tss0, strand, a0, a1).upper() == c['anchor_seq'].upper(), 'anchor self-check 失敗'
        else:
            print(f'[skip] {name}: 未知 method'); continue
    except Exception as e:
        print(f'[skip] {name}: {e}'); continue
    # 切 full window（不再分元件）
    a, b = ELEMENTS['full']
    full = slice_region(gseq, tss0, strand, a, b)
    bpm_best = scan_bpm_in_sequence(full)
    # record_id, not c['accession']: the CANDIDATES table keys on the filename
    # (NC_001416), while 03 records the versioned GenBank id (NC_001416.1).
    # Writing the versioned id here is what lets the two phage tables join.
    row = dict(name=name, phage=c['phage'], host=c['host'], sigma=c['sigma'],
               accession=record_id, strand=strand, tss_pos_1based=tss0+1, method=c['method'],
               location_ok=bool(c.get('location_ok', True)),
               full=full, source=c.get('source',''), notes=c.get('notes',''))
    row.update(bpm_best)
    rows.append(row)
    print(f'[ok]  {name}  full={full}')

print(f'\n完成 {len(rows)} 條')


[ok]  T7_A1  full=ATTTAAAATTTATCAAAAAGAGTATTGACTTAAAGTCTAACCTATAGGATACTTACAGCCATCGAGAGGGACACG
[ok]  T7_A2  full=ATAAGTCGCACGAAAAACAGGTATTGACAACATGAAGTAACATGCAGTAAGATACAAATCGCTAGGTAACACTAG
[ok]  T7_A3  full=GGCACATAAGGTGAAACAAAACGGTTGACAACATGAAGTAAACACGGTACGATGTACCACATGAAACGACAGTGA
[ok]  T7_B  full=GCGAGTGGCCTTTATGATTATCACTTTACTTATGAGGGAGTAATGTATATGCTTACTATCGGTCTACTCACCGCT
[ok]  T7_C  full=CTTACGCTCAACATTGATAAGCAACTTGACGCAATGTTAATGGGCTGATAGTCTTATCTTACAGGTCATCTGCGG
[ok]  T7_E[6]  full=TCGGTGATAACGGTCTTACGGATGATGATATTTACACATTACAGTGATATACTCAAGGCCACTACAGATAGTGGT
[ok]  T5_D_E_20  full=CTTGATAAAATATGACGGTCGGCACTTACCATAATGTGTCCCGCCCCCTAACTGGGATATAGCGGCCTGAGTGGA
[ok]  T5_H_22  full=GATGGTACTACTAAAAAATTGTTGACAATAGCCCAGCAATCGGTAAAATATCGATTTAGGCAGTACAAGAAAGGC
[ok]  T5_F_30  full=TATAATTACTTTATAAATTGATGAGAAGGAAACAAAATGAACAAAGTTGATAAAGCTCTAGTTTTCGCAGCAGCA
[ok]  T5_D_E_33  full=ACTAAAACTTAAAAATTTATTTGCTTAAATACTTAAACTTCTGTATAATACTTTCATAAATTAATGAGAGGAAGC
[ok]  T5_H_207  full=TAATTTTTAAAAAATTCATTTGCTAAA

In [7]:
# === self-check：full window 長度與內容快速檢視 ===
for r in rows:
    print(f"{r['name']}: len={len(r['full'])}  {r['full']}")


T7_A1: len=75  ATTTAAAATTTATCAAAAAGAGTATTGACTTAAAGTCTAACCTATAGGATACTTACAGCCATCGAGAGGGACACG
T7_A2: len=75  ATAAGTCGCACGAAAAACAGGTATTGACAACATGAAGTAACATGCAGTAAGATACAAATCGCTAGGTAACACTAG
T7_A3: len=75  GGCACATAAGGTGAAACAAAACGGTTGACAACATGAAGTAAACACGGTACGATGTACCACATGAAACGACAGTGA
T7_B: len=75  GCGAGTGGCCTTTATGATTATCACTTTACTTATGAGGGAGTAATGTATATGCTTACTATCGGTCTACTCACCGCT
T7_C: len=75  CTTACGCTCAACATTGATAAGCAACTTGACGCAATGTTAATGGGCTGATAGTCTTATCTTACAGGTCATCTGCGG
T7_E[6]: len=75  TCGGTGATAACGGTCTTACGGATGATGATATTTACACATTACAGTGATATACTCAAGGCCACTACAGATAGTGGT
T5_D_E_20: len=75  CTTGATAAAATATGACGGTCGGCACTTACCATAATGTGTCCCGCCCCCTAACTGGGATATAGCGGCCTGAGTGGA
T5_H_22: len=75  GATGGTACTACTAAAAAATTGTTGACAATAGCCCAGCAATCGGTAAAATATCGATTTAGGCAGTACAAGAAAGGC
T5_F_30: len=75  TATAATTACTTTATAAATTGATGAGAAGGAAACAAAATGAACAAAGTTGATAAAGCTCTAGTTTTCGCAGCAGCA
T5_D_E_33: len=75  ACTAAAACTTAAAAATTTATTTGCTTAAATACTTAAACTTCTGTATAATACTTTCATAAATTAATGAGAGGAAGC
T5_H_207: len=75  TAATTTTTAAAAAATTCATTTGCTAAACGCTTCAAATTCTCGTATAATATACTTCATAAA

In [8]:
# === BPM 掃 full sequence，並把能量與預測表現量加入 rows ===
for row in rows:
    bpm_best = scan_bpm_in_sequence(row['full'])
    row.update(bpm_best)
    print(f"[bpm] {row['name']} dG={row.get('bpm_dG_total')} pred_logexp={row.get('bpm_pred_logexp')}")



[bpm] T7_A1 dG=-6.900182599245759 pred_logexp=2.539982917714782
[bpm] T7_A2 dG=-7.14260761188458 pred_logexp=2.6432894635007416
[bpm] T7_A3 dG=-8.04823732346457 pred_logexp=2.8881800355267004
[bpm] T7_B dG=-6.811358591031806 pred_logexp=2.4983363817271402
[bpm] T7_C dG=-7.188183023635901 pred_logexp=2.660928653095131
[bpm] T7_E[6] dG=-6.791935899254084 pred_logexp=2.4889754071650017
[bpm] T5_D_E_20 dG=-4.720075347148077 pred_logexp=1.2581904548761054
[bpm] T5_H_22 dG=-8.234568453598708 pred_logexp=2.9146890066824183
[bpm] T5_F_30 dG=-6.320005412448591 pred_logexp=2.237455250968583
[bpm] T5_D_E_33 dG=-8.161452627347932 pred_logexp=2.9050426241430425
[bpm] T5_H_207 dG=-8.936740926531003 pred_logexp=2.970837503970138
[bpm] T5_N_25 dG=-8.161452627347932 pred_logexp=2.9050426241430425


[bpm] T5_N_26 dG=-8.161452627347932 pred_logexp=2.9050426241430425
[bpm] T5_K_28b dG=-9.137475115327351 pred_logexp=2.978752035086862
[bpm] T5_K_28a dG=-8.936740926531003 pred_logexp=2.970837503970138
[bpm] T5_G_25 dG=-8.982226595597089 pred_logexp=2.972849473499219
[bpm] T5_J_5 dG=-9.137475115327351 pred_logexp=2.978752035086862
[bpm] lambda_PR dG=-7.220119598714749 pred_logexp=2.672943240742041
[bpm] lambda_PL dG=-5.692833742576802 pred_logexp=1.8566227940395708
[bpm] L5_Pleft dG=-6.033803772725568 pred_logexp=2.0678804635234136
[bpm] D29_Pleft dG=-7.008520269917614 pred_logexp=2.58808188640925


In [9]:
# === 輸出 CSV ===
cols = ['name','phage','host','sigma','accession','strand','tss_pos_1based','method',
        'location_ok',
        'full',
        'bpm_core_start_in_full_1based','bpm_core_end_in_full_1based','bpm_spacer_len',
        'bpm_minus35','bpm_spacer','bpm_minus10','bpm_core_seq',
        'bpm_dG_minus35','bpm_dG_spacer','bpm_dG_minus10','bpm_dG_total',
        'bpm_pred_exp','bpm_pred_logexp',
        'source','notes']
df = pd.DataFrame(rows)[cols] if rows else pd.DataFrame(columns=cols)
out_csv = OUT / '04_phage_curated_raw.csv'  # intermediate; standardised table is written by the last cell
df.to_csv(out_csv, index=False)
print('已寫出', out_csv)
df


已寫出 C:\project\Whole-model\MS2_Data_PyTorch\scripts\library_release\outputs\04_phage_curated_raw.csv


,name,phage,host,sigma,accession,strand,tss_pos_1based,method,location_ok,full,...,bpm_minus10,bpm_core_seq,bpm_dG_minus35,bpm_dG_spacer,bpm_dG_minus10,bpm_dG_total,bpm_pred_exp,bpm_pred_logexp,source,notes
0,T7_A1,T7,E.coli,sigma70,NC_001604.1,+,498,manual,True,ATTTAAAATTTATCAAAAAGAGTATTGACTTAAAGTCTAACCTATA...,...,GATACT,TTGACTTAAAGTCTAACCTATAGGATACT,-3.162231,0.000000,-3.737952,-6.900183,346.723212,2.539983,Dunn&Studier 1983; 1978 early-promoter seq,major early promoter
1,T7_A2,T7,E.coli,sigma70,NC_001604.1,+,626,manual,True,ATAAGTCGCACGAAAAACAGGTATTGACAACATGAAGTAACATGCA...,...,TAAGAT,TTGACAACATGAAGTAACATGCAGTAAGAT,-3.937519,0.957271,-4.162359,-7.142608,439.834674,2.643289,1978 early-promoter seq,A2/A3 share 17bp in -35 region
2,T7_A3,T7,E.coli,sigma70,NC_001604.1,+,750,manual,True,GGCACATAAGGTGAAACAAAACGGTTGACAACATGAAGTAAACACG...,...,TACGAT,TTGACAACATGAAGTAAACACGGTACGAT,-3.937519,0.000000,-4.110718,-8.048237,773.000964,2.888180,1978 early-promoter seq,
3,T7_B,T7,E.coli,sigma70,NC_001604.1,+,1514,manual,True,GCGAGTGGCCTTTATGATTATCACTTTACTTATGAGGGAGTAATGT...,...,TATGCT,TTTACTTATGAGGGAGTAATGTATATGCT,-2.457432,0.000000,-4.353926,-6.811359,315.018734,2.498336,1978 early-promoter seq,
4,T7_C,T7,E.coli,sigma70,NC_001604.1,+,3113,manual,True,CTTACGCTCAACATTGATAAGCAACTTGACGCAATGTTAATGGGCT...,...,TAGTCT,TTGACGCAATGTTAATGGGCTGATAGTCT,-3.458144,0.000000,-3.730039,-7.188183,458.066628,2.660929,1978 early-promoter seq,
5,T7_E[6],T7,E.coli,sigma70,NC_001604.1,+,36836,manual,True,TCGGTGATAACGGTCTTACGGATGATGATATTTACACATTACAGTG...,...,TATACT,ATGATATTTACACATTACAGTGATATACT,-1.591980,0.000000,-5.199956,-6.791936,308.301336,2.488975,1978 early-promoter seq,
6,T5_D_E_20,T5,E.coli,sigma70,AY543070.1,-,4604,manual,False,CTTGATAAAATATGACGGTCGGCACTTACCATAATGTGTCCCGCCC...,...,TACCAT,TTGATAAAATATGACGGTCGGCACTTACCAT,-3.462334,2.871813,-4.129555,-4.720075,18.121346,1.258190,data/T5 promoter.xlsx; AY543070.1 GenBank regu...,promoter P-D/E 20; location=complement(4338..4...
7,T5_H_22,T5,E.coli,sigma70,AY543070.1,-,4813,manual,True,GATGGTACTACTAAAAAATTGTTGACAATAGCCCAGCAATCGGTAA...,...,TAAAAT,TTGACAATAGCCCAGCAATCGGTAAAAT,-3.937519,0.711340,-5.008389,-8.234568,821.654062,2.914689,data/T5 promoter.xlsx; AY543070.1 GenBank regu...,promoter P-H 22; location=complement(4799..487...
8,T5_F_30,T5,E.coli,sigma70,AY543070.1,-,16724,manual,False,TATAATTACTTTATAAATTGATGAGAAGGAAACAAAATGAACAAAG...,...,CAAAAT,TTTATAAATTGATGAGAAGGAAACAAAAT,-2.757535,0.000000,-3.562470,-6.320005,172.764796,2.237455,data/T5 promoter.xlsx; AY543070.1 GenBank regu...,promoter P-F 30; location=complement(16779..16...
9,T5_D_E_33,T5,E.coli,sigma70,AY543070.1,-,42241,manual,True,ACTAAAACTTAAAAATTTATTTGCTTAAATACTTAAACTTCTGTAT...,...,TATAAT,TTGCTTAAATACTTAAACTTCTGTATAAT,-2.641560,0.000000,-5.519893,-8.161453,803.604989,2.905043,data/T5 promoter.xlsx; AY543070.1 GenBank regu...,promoter P-D/E 33; location=complement(42227.....


## 注意事項
- **座標版本**：accession 一定要對到你實際下載的版本（RefSeq 改版時座標可能位移）。`tss_pos` 與 NCBI 顯示一致用 1-based，程式內部轉 0-based。
- **元件邊界是 first-pass**：`ELEMENTS` 的 -35/-10 假設 spacer≈17bp 的標準偏移；真實 promoter 要用你的 scanner 或人工 refine，不要直接當定論。
- **anchor 法最不依賴座標**：只要論文有給一段精確序列，貼進來 + 標起點座標即可，notebook 會自己定位並自我驗證。
- **M. smegmatis 之後接**：同一套表格與函式可直接用——換成 mycobacteriophage 的 accession 與你查到的 σA promoter TSS 即可，不用改程式。


## Standardised output for the assembly step

Writes `outputs/04_phage_promoters.csv`, schema-compatible with 03.

In [10]:
# === Standardised curated-phage table (04_phage_promoters.csv) ===
# Same 75 nt -60..+15 window as 03, so the two phage sources concatenate.
import pandas as pd

raw = pd.read_csv(require(RELEASE_OUT / "04_phage_curated_raw.csv", "curated phage output"))
print("curated rows:", len(raw))

RENAME = {
    "full": "promoter_sequence",
    "phage": "phage_species",
    "host": "phage_host",
    "sigma": "phage_sigma_factor",
    "accession": "phage_genome_accession",
    # 'manual' / 'anchor' per row - keep it instead of letting the constant
    # method label below overwrite how each +1 was actually pinned down.
    "method": "curation_method",
}
missing = [c for c in RENAME if c not in raw.columns]
if missing:
    raise KeyError(f"curated phage table missing columns: {missing}")

curated = raw.rename(columns=RENAME).copy()
curated["method"] = "literature_manual"
curated["phage_promoter_window"] = "-60..15"

# Two T5 features in the source sheet are not 75 bp, so the +1 inferred from
# their upstream edge is not trustworthy. location_ok carries that into qc_pass
# rather than leaving it in a notes string; 06 (INCLUDE_QC_FAIL=False) then
# leaves those rows out of the library.
if "location_ok" in curated.columns:
    location_ok = curated["location_ok"].fillna(True).astype(bool)
    if not location_ok.all():
        print("location_ok=False (qc_pass forced False):",
              curated.loc[~location_ok, "name"].tolist())
else:
    location_ok = None

out = standardize(curated, source="phage", candidate_id="PHG-C", extra_pass=location_ok)
out.to_csv(RELEASE_OUT / "04_phage_promoters.csv", index=False)

print(f"\nwrote {RELEASE_OUT / '04_phage_promoters.csv'}  {out.shape}")
print("promoter_length:", out["promoter_length"].value_counts().to_dict())
print("qc_pass:", int(out["qc_pass"].sum()), "/", len(out))
print(out[STD_COLS].head(5).to_string(index=False))

# 03 and 04 must line up on the shared columns, otherwise 06 would concat into
# a ragged table. Check here, where the mismatch is cheap to see.
p3 = RELEASE_OUT / "03_phage_promoters.csv"
if p3.exists():
    a = pd.read_csv(p3, nrows=1)
    common = [c for c in STD_COLS if c in a.columns and c in out.columns]
    print("\nshared columns with 03:", common)
    if len(common) != len(STD_COLS):
        raise ValueError("03 and 04 do not share the standard column set")
else:
    print("\n03_phage_promoters.csv not written yet - run 03 before 06.")


curated rows: 21
location_ok=False (qc_pass forced False): ['T5_D_E_20', 'T5_F_30']

wrote C:\project\Whole-model\MS2_Data_PyTorch\scripts\library_release\outputs\04_phage_promoters.csv  (21, 31)
promoter_length: {75: 21}
qc_pass: 19 / 21
candidate_id source                                                           promoter_sequence  promoter_length  alphabet_valid  qc_pass
  PHG-C00001  phage ATTTAAAATTTATCAAAAAGAGTATTGACTTAAAGTCTAACCTATAGGATACTTACAGCCATCGAGAGGGACACG               75            True     True
  PHG-C00002  phage ATAAGTCGCACGAAAAACAGGTATTGACAACATGAAGTAACATGCAGTAAGATACAAATCGCTAGGTAACACTAG               75            True     True
  PHG-C00003  phage GGCACATAAGGTGAAACAAAACGGTTGACAACATGAAGTAAACACGGTACGATGTACCACATGAAACGACAGTGA               75            True     True
  PHG-C00004  phage GCGAGTGGCCTTTATGATTATCACTTTACTTATGAGGGAGTAATGTATATGCTTACTATCGGTCTACTCACCGCT               75            True     True
  PHG-C00005  phage CTTACGCTCAACATTGATAAGCAACTTGACGCAATGTTAATGGGCTGATAG